# PDM on Colab (L4 — reduced-scope pilot) — patch-diffusion UAD pipeline

**This notebook's workflow:** upload `pdm_code.zip` + the **raw** BraTS dataset to
Drive; preprocess on Colab (reads NIfTI from Drive → fast local `/content`);
train; calibrate; evaluate; explain.

**Reduced scope (3-day / ~50-credit budget):** train a short `--epochs 6` pilot and
keep the best-val EMA checkpoint; protect credits for calibrate/evaluate/explain.

**Logs:** every stage streams live into its cell (`python -u`) **and** is saved
to `outputs/logs/<stage>_*.log` on Drive (survives disconnects).

> Runtime → Change runtime type → **L4 GPU**, then run cells top to bottom.
> ⚠️ `/content/processed` is ephemeral — after a disconnect, re-run the unzip +
> preprocess cells, then re-run train with `--resume` (it continues from the
> checkpoint on Drive).

## 1. Confirm the GPU (should say L4)

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch; print('torch', torch.__version__, '| cuda', torch.cuda.is_available(), '| bf16', torch.cuda.is_bf16_supported())

## 2. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Unzip the code into /content (edit DRIVE_DIR if needed)

In [ ]:
import os, time
DRIVE_DIR = '/content/drive/MyDrive/pdm'          # folder holding pdm_code.zip
CODE_ZIP  = f'{DRIVE_DIR}/pdm_code.zip'
assert os.path.exists(CODE_ZIP), f'missing {CODE_ZIP}'
t0 = time.time()
!mkdir -p /content/pdm_code && unzip -q -o '{CODE_ZIP}' -d /content/pdm_code
print(f'code unpacked in {time.time()-t0:.0f}s'); print(os.listdir('/content/pdm_code'))

## 4. Install deps + set all paths + enable diffusers logging
Installs only what Colab lacks (`diffusers`, `nibabel`) — **never reinstall
torch** (it would break the CUDA build). `DIFFUSERS_VERBOSITY=info` turns on
Hugging Face diffusers' own logs (they are silenced by default).

In [ ]:
%cd /content/pdm_code
!pip install -q -U diffusers nibabel
# Optional CRF refinement (pipeline runs fine without it):
!pip install -q git+https://github.com/lucasb-eyer/pydensecrf.git || echo 'CRF optional - skipped'

# Paths: raw data + fast local processed dir + persistent Drive outputs.
%env PDM_DATA_ROOT=/content/drive/MyDrive/BraTS-PEDs-v1/Training
%env PDM_PROCESSED_ROOT=/content/processed
%env PDM_OUTPUT_ROOT=/content/drive/MyDrive/pdm/outputs
# Enable Hugging Face diffusers' own INFO logs (silenced by default):
%env DIFFUSERS_VERBOSITY=info
!mkdir -p /content/drive/MyDrive/pdm/outputs
import torch; print('torch', torch.__version__, '| cuda', torch.cuda.is_available())

## 5. Preprocess on Colab (~15–40 min, reads NIfTI from Drive)
`-u` = unbuffered so progress streams live. Also saved to `outputs/logs/preprocess_*.log`.

In [ ]:
!python -u scripts/00_preprocess.py --splits splits

## 6. Verify the dataset (must print VERIFICATION PASSED)
Checks patient leakage, mask validity, intensity ranges; writes
`outputs/results/dataset_check.png` montage for a visual sanity check.

In [ ]:
!python -u scripts/verify_dataset.py

## 7. Train (L4, bf16) — short pilot, live logs + resume-safe
Logs stream here **and** to `outputs/logs/train_*.log` on Drive. After a
disconnect: re-run cells 3+5 (unzip+preprocess), then re-run this cell **with
`--resume` appended** — it continues from the last checkpoint.

In [ ]:
# Reduced-scope pilot on L4 (3-day / ~50-credit budget). Fresh run: warmup_epochs=1
# in config so the cosine LR fully decays within these few epochs. Take the best-val
# EMA checkpoint wherever credits/time stop you (~3-5 epochs is plenty of steps).
# If the session disconnects MID-RUN, re-run cells 3+5 then re-run THIS cell with
# --resume appended to continue from the last Drive checkpoint.
# If L4 OOMs at --bs 256, drop to --bs 128.
!python -u scripts/01_train.py --epochs 6 --bs 256 --noise simplex

## (optional) Peek at the saved training log file on Drive

In [ ]:
import glob, os
logs = sorted(glob.glob('/content/drive/MyDrive/pdm/outputs/logs/train_*.log'), key=os.path.getmtime)
print('latest log:', logs[-1] if logs else 'none')
if logs:
    !tail -n 40 '{logs[-1]}'

## 8. Calibrate the anomaly threshold (healthy val slices)

In [ ]:
!python -u scripts/02_calibrate.py --max-samples 150

## 9. Evaluate (AUROC / AUPRC / DICE + panels)

In [ ]:
!python -u scripts/03_evaluate.py --n-images 80 --max-plots 24
import json
m = json.load(open('/content/drive/MyDrive/pdm/outputs/results/metrics.json'))
print({k: m[k] for k in ['auroc_mean','auprc_mean','dice_calibrated','dice_best_global','dice_oracle']})

## 10. XAI — counterfactual + attribution panels

In [ ]:
!python -u scripts/04_explain.py --n-cases 6

## 11. Results (all on Drive)
`MyDrive/pdm/outputs/`: `checkpoints/`, `logs/`, `results/metrics.json`,
`results/eval_*.png`, `xai/*.png`. Download for your report.